### 1. Data Ingestion

In [ ]:
## 7 CSV Files --> 7 DFs

In [ ]:
import pandas as pd

In [ ]:
df_Author = pd.read_csv('./Dataset/Author.csv')
df_Author_Book = pd.read_csv('./Dataset/Author_Book.csv')
df_Book = pd.read_csv('./Dataset/Book.csv')
df_Book_Order = pd.read_csv('./Dataset/Book_Order.csv')
df_Category = pd.read_csv('./Dataset/Category.csv')
df_Customer = pd.read_csv('./Dataset/Customer.csv')
df_Ordering = pd.read_csv('./Dataset/Ordering.csv')

### 2. Bronze Layer

In [ ]:
from sqlalchemy import create_engine

engine = create_engine(
    "mssql+pyodbc://<YOUR_SERVER>/Book_dwh"
    "?trusted_connection=yes"
    "&driver=ODBC+Driver+17+for+SQL+Server"
    "&TrustServerCertificate=yes"
)

In [ ]:
def insert_bronze(df , table_name):
    df.to_sql(
    table_name,
    con=engine,
    schema = 'bronze',
    index=False,
    if_exists='append'
)

In [ ]:
insert_bronze(df_Author , 'author')

In [ ]:
insert_bronze(df_Author_Book, 'author_book')

In [ ]:
insert_bronze(df_Book, 'book')

In [ ]:
insert_bronze(df_Book_Order, 'book_order')

In [ ]:
insert_bronze(df_Category, 'category')

In [ ]:
insert_bronze(df_Customer, 'customer')

In [ ]:
insert_bronze(df_Ordering, 'ordering')

### 3. Silver Layer

In [ ]:
from sqlalchemy import create_engine

engine = create_engine(
    "mssql+pyodbc://<YOUR_SERVER>/Book_dwh"
    "?trusted_connection=yes"
    "&driver=ODBC+Driver+17+for+SQL+Server"
    "&TrustServerCertificate=yes"
)

In [ ]:
df_Author = pd.read_sql('select * from bronze.author', con=engine)
df_Author_Book = pd.read_sql('select * from bronze.author_book', con=engine)
df_Book = pd.read_sql('select * from bronze.book', con=engine)
df_Book_Order = pd.read_sql('select * from bronze.book_order', con=engine)
df_Category = pd.read_sql('select * from bronze.category', con=engine)
df_Customer = pd.read_sql('select * from bronze.customer', con=engine)
df_Ordering = pd.read_sql('select * from bronze.ordering', con=engine)

In [ ]:
df_Author.dtypes

In [ ]:
df_Author['AuthorID'] = df_Author['AuthorID'].astype('int64')

In [ ]:
df_Author.dtypes

In [ ]:
df_Author.isnull().sum()

In [ ]:
df_Author_Book.dtypes

In [ ]:
df_Author_Book['AuthorID'] = df_Author_Book['AuthorID'].astype('int64')
df_Author_Book['BookID'] = df_Author_Book['BookID'].astype('int64')

In [ ]:
df_Author_Book.dtypes

In [ ]:
df_Author_Book.isnull().sum()

In [ ]:
df_Book.dtypes

In [ ]:
df_Book['BookID'] = df_Book['BookID'].astype('int64')
df_Book['CategoryID'] = df_Book['CategoryID'].astype('int64')
df_Book['Year'] = df_Book['Year'].astype('int64')
df_Book['Price'] = df_Book['Price'].astype('float64')
df_Book['NoPages'] = df_Book['NoPages'].astype('int64')


In [ ]:
df_Book.isnull().sum()

In [ ]:
df_Book_Order.isnull().sum()

In [ ]:
df_Book_Order['CustomerID'] = df_Book_Order['CustomerID'].fillna(0).astype('int64')

In [ ]:
df_Book_Order.dtypes

In [ ]:
df_Book_Order['OrderID'] = df_Book_Order['OrderID'].astype('int64')
df_Book_Order['OrderDate'] = pd.to_datetime(df_Book_Order['OrderDate'], format='%Y-%m-%d')

In [ ]:
df_Book_Order.dtypes

In [ ]:
df_Book_Order.isnull().sum()

In [ ]:
df_Category.dtypes

In [ ]:
df_Category['CategoryID'] = df_Category['CategoryID'].astype('int64')

In [ ]:
df_Category.isnull().sum()

In [ ]:
df_Customer.dtypes

In [ ]:
df_Customer.isnull().sum()

In [ ]:
df_Customer['FirstName'] = df_Customer['FirstName'].fillna('Unknown')
df_Customer['LastName'] = df_Customer['LastName'].fillna('Unknown')
df_Customer['City'] = df_Customer['City'].fillna('Unknown')

In [ ]:
df_Customer['CustomerID'] = df_Customer['CustomerID'].astype('int64')
df_Customer['ZipCode'] = df_Customer['ZipCode'].astype('int64')

In [ ]:
df_Customer['FullName'] = df_Customer['FirstName'] + ' ' + df_Customer['LastName']

In [ ]:
df_Customer.dtypes

In [ ]:
df_Customer.isnull().sum()

In [ ]:
df_Ordering.dtypes

In [ ]:
df_Ordering.isnull().sum()

In [ ]:
df_Ordering['Quantity'] = df_Ordering['Quantity'].fillna(1).astype('int64')

In [ ]:
df_Ordering['OrderID'] = df_Ordering['OrderID'].astype('int64')
df_Ordering['BookID'] = df_Ordering['BookID'].astype('int64')

In [ ]:
df_Ordering_Book = df_Ordering.merge(df_Book, on='BookID', how='left')

In [ ]:
df_Ordering_Book

In [ ]:
df_Ordering_Book = df_Ordering_Book [['OrderID' , 'BookID' , 'Quantity' , 'Price_y']]

In [ ]:
df_Ordering_Book.dtypes

In [ ]:
df_Ordering_Book.isnull().sum()

In [ ]:
df_Ordering_Book ['Price'] = df_Ordering_Book ['Price_y'] 

In [ ]:
df_Ordering_Book ['Quantity'] = df_Ordering_Book ['Quantity'].astype('float64')

In [ ]:
df_Ordering_Book.columns

In [ ]:
df_Ordering_Book.drop(['Price_y'], axis=1, inplace=True)

In [ ]:
def insert_silver(df , table_name):
    df.to_sql(
    table_name,
    con=engine,
    schema = 'silver',
    index=False,
    if_exists='append'
)

In [ ]:
df_Author.dtypes

In [ ]:
insert_silver(df_Author , 'author')

In [ ]:
df_Author_Book.dtypes

In [ ]:
insert_silver(df_Author_Book , 'author_book')

In [ ]:
df_Book.dtypes

In [ ]:
insert_silver(df_Book , 'book')

In [ ]:
df_Book_Order.dtypes

In [ ]:
insert_silver(df_Book_Order , 'book_order')

In [ ]:
df_Category.dtypes

In [ ]:
insert_silver(df_Category , 'category')

In [ ]:
df_Customer.dtypes

In [ ]:
insert_silver(df_Customer , 'customer')

In [ ]:
df_Ordering_Book.dtypes

In [ ]:
insert_silver(df_Ordering_Book , 'ordering')

### 4. Gold Layer

In [ ]:
from sqlalchemy import create_engine

engine = create_engine(
    "mssql+pyodbc://<YOUR_SERVER>/Book_dwh"
    "?trusted_connection=yes"
    "&driver=ODBC+Driver+17+for+SQL+Server"
    "&TrustServerCertificate=yes"
)

In [ ]:
df_author_book = pd.read_sql('select * from silver.author_book', con=engine)
df_author = pd.read_sql('select * from silver.author', con=engine)
df_book_order = pd.read_sql('select * from silver.book_order', con=engine)
df_book = pd.read_sql('select * from silver.book', con=engine)
df_category = pd.read_sql('select * from silver.category', con=engine)
df_customer = pd.read_sql('select * from silver.customer', con=engine)
df_ordering = pd.read_sql('select * from silver.ordering', con=engine)

In [ ]:
customer_dim = df_customer[['CustomerID','FirstName','LastName','FullName', 'City']]

In [ ]:
customer_dim

In [ ]:
def insert_gold(df , table_name):
    df.to_sql(
    table_name,
    con=engine,
    schema = 'gold',
    index=False,
    if_exists='append'
)

In [ ]:
insert_gold(customer_dim , 'customer_dim')

In [ ]:
book_dim = pd.merge(df_book, df_category, on='CategoryID', how='left')

In [ ]:
book_dim.columns

In [ ]:
book_dim = book_dim[['BookID', 'Title', 'ISBN' , 'Year' , 'Price', 'NoPages', 'BookDescription' , 'CategoryDescription']]

In [ ]:
book_dim

In [ ]:
insert_gold(book_dim , 'book_dim')

In [ ]:
author_dim = pd.merge(df_author_book, df_author, on='AuthorID', how='left')

In [ ]:
author_dim

In [ ]:
insert_gold(author_dim , 'author_dim')

In [ ]:
fact_order = pd.merge(df_book_order, df_ordering, on='OrderID', how='left')

In [ ]:
fact_order

In [ ]:
fact_order['TotalAmount'] = (fact_order['Quantity'] * fact_order['Price']).astype('float64')

In [ ]:
book_dim = pd.read_sql('select * from gold.book_dim', con=engine)
author_dim = pd.read_sql('select * from gold.author_dim', con=engine)
customer_dim = pd.read_sql('select * from gold.customer_dim', con=engine)

In [ ]:
fact_order = pd.merge(fact_order, book_dim, on='BookID', how='left')
fact_order = pd.merge(fact_order, author_dim, on='BookID', how='left')
fact_order = pd.merge(fact_order, customer_dim, on='CustomerID', how='left')

In [ ]:
fact_order.columns

In [ ]:
fact_order = fact_order[['OrderID', 'author_Sk' ,'book_sk' ,'Customer_sk', 'Quantity', 'Price_x', 'TotalAmount']]

In [ ]:
fact_order['Price'] = fact_order['Price_x']

In [ ]:
fact_order.drop(['Price_x'], axis=1, inplace=True)

In [ ]:
fact_order

In [ ]:
insert_gold(fact_order , 'fact_orders')